## OLCI Sargassum Detection
`Add SNAP pin exports to the per-pixel feature database`

### 1. Import Libraries

In [ ]:
import os

import numpy as np
import pandas as pd

# ignore warnings
import warnings
warnings.filterwarnings('ignore')

### 2. Define inputs

List only the sheets that are not yet in the database.

In [ ]:
pin_file    = 'Book1.xlsx'
sheet_names = ['Sheet4']
db_path     = 'sargassum_feature_db.csv'

### 3. Read the SNAP pin export sheets

In [ ]:
frames = []
for sheet_name in sheet_names:
    raw = pd.read_excel(pin_file, sheet_name = sheet_name, header = None)

    # product name sits in the '# Product:' row of the SNAP header block
    product_row = raw[raw[0].astype(str).str.startswith('# Product')].index[0]
    product = str(raw.iloc[product_row, 1]).removesuffix('.SEN3')

    # column header row is the one starting with 'Name'
    header_row = raw[raw[0] == 'Name'].index[0]
    df = pd.read_excel(pin_file, sheet_name = sheet_name, header = header_row)

    # SNAP leaves the label column unnamed
    df = df.rename(columns = {df.columns[6]: 'label'})

    # wavelength headers are floats; rename to band-centre strings
    band_cols = [c for c in df.columns if isinstance(c, (int, float))]
    df = df.rename(columns = {c: f'L_{c:g}' for c in band_cols})

    df['sheet']   = sheet_name
    df['product'] = product
    frames.append(df)
    print(f"{sheet_name}: {len(df)} pins from {product}")

new = pd.concat(frames, ignore_index = True)

### 4. Clean labels and drop repeated pixels

X/Y are pixel positions within one scene, so repeated pins are tested on `product` + `X` + `Y`.

OLCI fills some pixels (`duplicated` flag) by copying a neighbour, geolocation included. Those copies are dropped by testing `product` + `Lon` + `Lat`, so each measurement enters the database once.

In [ ]:
new['label'] = new['label'].str.strip().str.lower()
new[['SGI', 'AFAI']] = new[['SGI', 'AFAI']].apply(pd.to_numeric)
new['is_sargassum'] = (new['label'] == 'sargassum').astype(int)

n_before = len(new)
new = new.drop_duplicates(subset = ['product', 'X', 'Y'])
print(f"Dropped {n_before - len(new)} repeated pins within the new sheets")

# duplicated pixels share their neighbour's lon/lat and radiances exactly
n_before = len(new)
new = new.drop_duplicates(subset = ['product', 'Lon', 'Lat'])
print(f"Dropped {n_before - len(new)} duplicated-pixel copies within the new sheets")

band_cols = [c for c in new.columns if str(c).startswith('L_')]
new = new[['Name', 'sheet', 'product', 'X', 'Y', 'Lon', 'Lat', 'label', 'is_sargassum'] + band_cols + ['SGI', 'AFAI']]

### 5. Append to the database and save

In [ ]:
if os.path.exists(db_path):
    db = pd.read_csv(db_path)
    assert 'product' in db.columns, 'database has no product column'
    db = pd.concat([db, new[db.columns]], ignore_index = True)
else:
    db = new

# pixels already in the database keep their original row
n_before = len(db)
db = db.drop_duplicates(subset = ['product', 'X', 'Y'])
db = db.drop_duplicates(subset = ['product', 'Lon', 'Lat']).reset_index(drop = True)
print(f"Dropped {n_before - len(db)} pixels already in the database")

db.to_csv(db_path, index = False)
print(f"{db_path}: {len(db)} pixels")
pd.crosstab(db['product'].str[16:31], db['label'])